# Blur

Quantum Blur turns an image into a single quantum state, rotates every qubit once, and measures the state back into pixels. One gate moves amplitude between pixels that can be far apart, so instead of soft focus you get echoes and interference that no convolution kernel produces.

## Setup

In [ ]:
import getpass
import os

from moth_atlas import Atlas

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY
ENGINE = "blur-v1"

In [ ]:
from IPython.display import Image, display


def show(images, width=200):
    """Display {label: image bytes}."""
    for label, data in images.items():
        print(label)
        display(Image(data, width=width))

## Try it

Every engine follows the same path: **upload inputs → submit a job → wait → download the output.**

### 1. Pick an image

Point `IMAGE` at your own image, or keep the sample cat beside this notebook.

In [ ]:
IMAGE = "cat.png"

image = open(IMAGE, "rb").read()
show({"input": image})

### 2. Upload the asset

`upload` wraps the three asset calls (create → PUT → complete).

In [ ]:
asset = client.assets.upload(IMAGE)
print("asset", asset.asset_id)

### 3. Blur the asset

`submit` starts a job on the engine; `wait` polls until it completes.

In [ ]:
params = {"strength": 0.5, "style": "rx", "reach": 0.0, "size": 1024, "downscale": True}

job = client.jobs.submit(ENGINE, params=params, input_files={"image": asset.asset_id})
client.jobs.wait(job.job_id, on_update=lambda s: print(s.status))
print("job", job.job_id)

### 4. Download the result

In [ ]:
output = client.jobs.download(job.job_id)
show({"input": image, f"strength {params['strength']}": output})

Steps 2–4 in one call: `client.jobs.run(ENGINE, params={...}, files={"image": IMAGE})`, then `download`. The experiments below use it.

## Parameters

| name | type | default | what it does |
|---|---|---|---|
| `image` (input file) | PNG / JPEG / WebP / TIFF / BMP, any mode | required | blurred one RGB channel at a time; alpha is passed through |
| `mask` (input file) | image | none | blurs each connected region of the mask on its own and blends by mask intensity; without one, the alpha channel is used, else the whole canvas |
| `strength` | float, 0 to 1 | 0.5 | scales every rotation; 0 returns the input unchanged |
| `style` | `rx` or `ry` | `rx` | `rx` spreads echoes across the image, `ry` concentrates them on edges |
| `reach` | float, 0 to 1 | 0 | 0 weights each angle by local brightness, 1 turns every qubit equally |
| `size` | int, 8 to 1024 | 1024 | pixel budget per pass: $2\lceil\log_2 \text{size}\rceil$ qubits |
| `downscale` | bool | true | a region over budget is downscaled (true) or tiled (false), never the whole image |

**Returns:** the blurred image, in the same format and size as the input.

## How it works

Quantum Blur has no convolution kernel. The image is encoded as a single quantum state, each qubit is rotated once, and the state is measured back into pixels. Because a single gate redistributes amplitude between pixels that may be far apart on the canvas, the output shows interference patterns rather than soft focus.

**Encode.** Pixel $(x,y)$ with brightness $h$ maps to a basis state, Gray-coded per axis:

$$|\psi\rangle \;\propto\; \sum_{x,y} \sqrt{h(x,y)}\;|g(x)\,g(y)\rangle, \qquad n = \lceil\log_2 W\rceil + \lceil\log_2 H\rceil \text{ qubits}$$

**Rotate.** One `rx` (or `ry`) per qubit, with an image-dependent angle:

$$\theta_q = \pi\,\xi\,\big[(1-\text{reach})\, r_q + \text{reach}\big], \qquad \xi = \text{strength} \;(\texttt{rx})\;\text{ or }\;\text{strength}^2 \;(\texttt{ry})$$

where $r_q$ is the normalised brightness on neighbour pairs whose labels differ in qubit $q$.

**Read back.** $h' = |\text{amp}|^2 / \max$, rescaled to the channel's original maximum. With a mask $m$, the output is $m\cdot\text{blurred} + (1-m)\cdot\text{original}$.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Strength

$\xi$ scales every angle. At 0 the round trip is exact; the echoes grow with it up to the limit of 1 (past that the rotations would start to undo themselves, which is why the engine stops there).

In [ ]:
outputs = {}
for strength in (0.0, 0.25, 0.5, 0.75, 1.0):
    job = client.jobs.run(ENGINE, params={"strength": strength}, files={"image": IMAGE})
    outputs[f"strength {strength}"] = client.jobs.download(job.job_id)
show(outputs, width=170)

### Reach and style

`reach` lifts the locality: at 0 the angle on each qubit is weighted by the brightness it separates, at 1 every qubit turns by the same $\pi\xi$ and echoes land anywhere on the canvas. `ry` keeps the same angles (with $\xi = \text{strength}^2$) but its interference terms cancel instead of spreading, so weight collects at a few pixels and smooth areas go dark.

In [ ]:
outputs = {}
for style in ("rx", "ry"):
    for reach in (0.0, 0.5, 1.0):
        job = client.jobs.run(ENGINE, params={"strength": 0.5, "style": style, "reach": reach}, files={"image": IMAGE})
        outputs[f"{style} · reach {reach}"] = client.jobs.download(job.job_id)
show(outputs, width=170)

### Mask

A feathered disc: 1 inside, a soft ramp, 0 outside. The engine blurs the disc's bounding box as its own state and blends by mask intensity, so pixels outside the mask must come back untouched and the edge must be a gradient, not a seam.

In [ ]:
import io

from PIL import Image as PILImage, ImageDraw, ImageFilter

# A feathered disc in the middle of the image: white blurs, black stays untouched.
w, h = PILImage.open(IMAGE).size
r = 0.3 * min(w, h)
mask = PILImage.new("L", (w, h), 0)
ImageDraw.Draw(mask).ellipse((w / 2 - r, h / 2 - r, w / 2 + r, h / 2 + r), fill=255)
buf = io.BytesIO()
mask.filter(ImageFilter.GaussianBlur(r / 6)).save(buf, "PNG")
mask_png = buf.getvalue()

job = client.jobs.run(ENGINE, params={"strength": 0.8}, files={"image": IMAGE, "mask": ("mask.png", mask_png, "image/png")})
show({"input": image, "mask": mask_png, "masked": client.jobs.download(job.job_id)})

## Questions

**Is this running on real quantum hardware?**

No. Blur runs on a classical statevector simulator, so jobs run on CPU or GPU. The circuits and the interference are real; the execution is emulated. That keeps results exact and repeatable with no shot noise. Real hardware would need very high connectivity and enough shots to rebuild the whole state, which is not yet practical at image scale.

**Why does it look different from a classical blur?**

A classical blur averages each pixel with a fixed neighbourhood. Blur has no kernel: one rotation per qubit mixes amplitude between every pixel at once, so the shape of the smear comes from the circuit, not a window. That is where the echo-like, non-local look comes from.

**How does image size affect run time?**

Qubits grow with the log of the image size (16 for 256 × 256, 20 for 1024 × 1024), but simulator memory and time grow exponentially with qubits. Regions over the qubit budget are downscaled or tiled, depending on `downscale`.

**What happens with colour images?**

Each of red, green and blue is blurred as its own state and the channels are recombined. The same technique was used on ILĀ's [Recurse album](https://infinite.mothquantum.com).

**Why does strength 0 round-trip exactly?**

Encoding normalises the state, which throws away absolute brightness. The engine keeps each channel's original maximum and restores it on read-back, so with no rotation the output equals the input.

**Where can I read more?**

The open-source [QuantumBlur repository](https://github.com/qiskit-community/QuantumBlur) walks through the underlying library.